# OpenEnv + GRPO Integration (Multi-Turn RL)

In Module 2, we saw GRPO on single-turn tasks (Prompt $\to$ 8 generations $\to$ Math Verifier $\to$ Advantage).

Now we integrate OpenEnv with GRPO to train multi-turn agents.

## Bridging TRL/GRPO with Interactive Environments

This section connects the single-turn GRPO workflow to the stateful, multi-turn world of OpenEnv.

## 1. Architectural Shift: Single-Turn vs. Multi-Turn GRPO

### Single-Turn GRPO

Single-turn GRPO is a stateless batch generation setup:

```text
Prompt q
  → [LLM samples G outputs in parallel]
  → [Verifier scores G outputs]
  → [Advantage / Loss]
```

### Multi-Turn GRPO with OpenEnv

Multi-turn GRPO adds stateful, interactive rollout pipelines:

```text
Prompt q
  → Spin up G isolated OpenEnv environments: [Env_1, Env_2, ..., Env_G]
  → Turn 1: LLM samples Act_1 for all G envs → step all G envs in parallel
  → Turn 2: LLM reads Obs_1, samples Act_2 → step all G envs in parallel
  → ...
  → Turn H: all G envs reach Done → collect cumulative returns G_i
  → Compute relative advantage
  → Mask environment tokens
  → Compute policy-gradient loss
```

## 2. Group Rollout Life-Cycle in OpenEnv + GRPO

### Step 1: Environment Pool Initialization

We initialize 4 independent, isolated environment instances such as 4 sandbox containers or isolated working directories:

$$
\mathcal{E} = [\text{Env}_0, \text{Env}_1, \text{Env}_2, \text{Env}_3]
$$

Each environment executes `reset()` with the same initial task definition.

### Step 2: Multi-Step Interaction Loop

While at least one environment is still active:

- format the current context for all active environments into a prompt batch
- let the LLM policy generate the next action tokens for all active streams
- parse the generated actions into structured commands
- call `env.step(action)` concurrently across the environment pool
- append observations to the corresponding trajectory buffers
- mark an episode as complete when it returns `terminated=True` or `truncated=True`

### Step 3: Trajectory Flattening and Mask Construction

Once all 4 episodes finish, each trajectory $\tau_i$ is serialized into token IDs with its corresponding binary loss mask $\mathbf{M}_i$:

$$
\tau_i \to (\mathbf{x}_i, \mathbf{M}_i, R_i)
$$

Where:

- $\mathbf{x}_i$: the concatenated token sequence of all turns
- $\mathbf{M}_i$: $1$ on policy-generated tokens and $0$ on environment observations and prompt text
- $R_i$: the total scalar cumulative reward accumulated over the multi-turn run

### Step 4: Group Advantage Normalization

Like standard GRPO, we compute the relative advantage across the group:

$$
A_i = \frac{R_i - \text{mean}(\{R_0, \dots, R_{G-1}\})}{\text{std}(\{R_0, \dots, R_{G-1}\}) + \epsilon}
$$

### Step 5: Batched Parallel Backward Pass

We pass the 4 flattened trajectories through the Transformer in a single forward pass, compute log-probabilities, multiply by $\mathbf{M}_i \times A_i$, and backpropagate.

## 3. Key Engineering Bottleneck: Asynchronous Episode Horizons

In single-turn GRPO, all rollouts finish at roughly the same time. In multi-turn settings, different environments can take very different numbers of steps.

### Example

```text
Env 0 (Fast success): [Turn 1] → [Turn 2] → [Turn 3] → Done
Env 1 (Slow failure): [Turn 1] → [Turn 2] → [Turn 3] → ... → [Turn 10 (Truncated)]
```

### Production strategies to handle step variance

- **Dynamic batch masking / early exit**: remove finished environments from the active generation batch so the LLM does not waste compute on completed episodes.
- **Trajectory packing**: pack variable-length trajectories with padded batches or FlashAttention variable-length sequence tensors (`cu_seqlens`) to avoid wasting VRAM on padding tokens.

## Strategy 1: Dynamic Batch Masking

### Problem

Suppose your group size is $G = 4$. You pass all 4 environments into the LLM together in a batch:

```text
Turn 1: LLM generates actions for Env 0, Env 1, Env 2, Env 3 (batch size = 4)
Turn 2: Env 0 immediately succeeds (done = True)
```

### Naive approach

You keep the batch size at 4 and continue generating tokens for Env 0 even though it is already done. That wastes GPU compute.

### Efficient approach

On the next turn, remove Env 0 from the active generation list. The LLM then runs on the remaining active environments only.

```text
Turn 1: [Env 0, Env 1, Env 2, Env 3]
Turn 2: [Env 0 (DONE), Env 1, Env 2, Env 3]
Turn 3: [Env 1, Env 2, Env 3]
Turn 4: [Env 1 (DONE), Env 2, Env 3]
Turn 5: [Env 2, Env 3]
```

As environments finish, the active batch shrinks until all $G$ environments are done.

## Strategy 2: Trajectory Packing

### Problem

Now all 4 runs are finished. You need to feed their recorded token histories into the GPU at the same time to compute gradients.

The trajectories have different lengths:

- Trajectory 0: 200 tokens
- Trajectory 1: 1,600 tokens
- Trajectory 2: 1,800 tokens
- Trajectory 3: 1,800 tokens

### Naive way: standard tensor padding

In basic PyTorch, the batch must be a perfect rectangle:

```text
[Batch, Max_Length] → [4, 1800]
```

That forces padding tokens into shorter sequences and wastes GPU memory and compute.

### Production way: trajectory packing with FlashAttention varlen

Instead of adding fake padding tokens, flatten all real tokens into one continuous array:

$$
\text{Packed Tokens} = [\underbrace{200 \text{ tokens}}_{\text{Traj 0}}, \underbrace{1600 \text{ tokens}}_{\text{Traj 1}}, \underbrace{1800 \text{ tokens}}_{\text{Traj 2}}, \underbrace{1800 \text{ tokens}}_{\text{Traj 3}}]
$$

You also pass a metadata array called `cu_seqlens` so the GPU knows where each trajectory starts and ends.

$$
\text{cu\_seqlens} = [0, 200, 1800, 3600, 5400]
$$

This lets FlashAttention process all trajectories in one batch with zero wasted padding tokens.

## 4. How the Mean and Advantage Are Computed in a Multi-Turn Group

The main point is simple:

- the mean and standard deviation are computed over the episode rewards,
- not over the token lengths, logits, or hidden states.

### Example with $G = 4$ trajectories

Assume the four trajectories finish with the following scalar rewards:

- Trajectory 0: $R_0 = 1.0$
- Trajectory 1: $R_1 = 1.0$
- Trajectory 2: $R_2 = 0.0$
- Trajectory 3: $R_3 = 0.0$

### Step 1: Compute the group mean

$$
\mu = \frac{R_0 + R_1 + R_2 + R_3}{G}
= \frac{1.0 + 1.0 + 0.0 + 0.0}{4}
= \mathbf{0.5}
$$

### Step 2: Compute the group standard deviation

$$
\sigma = \sqrt{\frac{(1.0-0.5)^2 + (1.0-0.5)^2 + (0.0-0.5)^2 + (0.0-0.5)^2}{4}}
= \mathbf{0.5}
$$

### Step 3: Compute the scalar advantage for each trajectory

$$
A_i = \frac{R_i - \mu}{\sigma + \epsilon}
$$

So:

$$
A_0 = \frac{1.0 - 0.5}{0.5} = \mathbf{+1.0}
$$

$$
A_1 = \frac{1.0 - 0.5}{0.5} = \mathbf{+1.0}
$$

$$
A_2 = \frac{0.0 - 0.5}{0.5} = \mathbf{-1.0}
$$

$$
A_3 = \frac{0.0 - 0.5}{0.5} = \mathbf{-1.0}
$$

### Step 4: Apply the scalar advantage to each trajectory

Now each trajectory receives one scalar advantage value, which is broadcast to all policy tokens inside that trajectory:

- Trajectory 0: multiply all its policy tokens by $A_0 = +1.0$
- Trajectory 1: multiply all its policy tokens by $A_1 = +1.0$
- Trajectory 2: multiply all its policy tokens by $A_2 = -1.0$
- Trajectory 3: multiply all its policy tokens by $A_3 = -1.0$

The important takeaway is that trajectory length does not change the reward mean or the advantage computation. Each trajectory gets one score regardless of whether it used 200 tokens or 2,000 tokens.

## 5. Why Standard PyTorch Padding Is Expensive

### Problem

Standard PyTorch expects dense, rectangular tensors. If you have sequences of different lengths, it must pad them to a common width.

For example, with three sequences:

- Sequence 0: 3 tokens
- Sequence 1: 5 tokens
- Sequence 2: 2 tokens

PyTorch needs a rectangular batch like this:

```text
[3, 5, D]
```

That means introducing fake padding tokens such as `<PAD>` to fill the shorter rows.

### Why this is inefficient

Even if the padding tokens are masked out, the GPU still performs the underlying matrix operations for them. This wastes:

- memory bandwidth,
- compute cycles,
- and VRAM capacity.

### Why not just loop over trajectories one by one?

You could run each trajectory in a separate forward pass, but that creates another bottleneck:

- each small kernel launch has overhead,
- small batches underutilize GPU cores,
- and the CPU spends too much time scheduling work.

So the challenge is to process many variable-length sequences together without wasting compute on padding.

## 6. How FlashAttention Varlen Solves This

FlashAttention introduces a variable-length path that avoids rectangular padding.

### Step A: Pack real tokens into one flattened array

Instead of building a padded matrix, we concatenate only the real tokens from all sequences.

$$
\text{Packed Input} = [t_0, t_1, t_2, t_3, t_4, t_5, t_6, t_7, t_8, t_9]
$$

### Step B: Use cumulative sequence lengths

We also provide a small metadata array called `cu_seqlens`:

$$
\text{cu\_seqlens} = [0, 3, 8, 10]
$$

This tells the GPU where each sequence starts and ends.

### Step C: Attention only within each real sequence

The FlashAttention kernel uses those offsets to ensure attention is computed only inside the valid range of each trajectory.

### Result

This gives:

- zero wasted padding tokens,
- zero wasted attention compute on fake positions,
- and full GPU saturation through a single batched kernel launch.

## 7. Final Takeaway

The key idea is that multi-turn GRPO is not just a bigger version of single-turn GRPO.

It requires:

1. episodic rewards instead of one-shot verifier scores,
2. per-trajectory advantage values,
3. masked policy-token updates,
4. and variable-length batch handling for efficiency.

That is why training multi-turn agents with OpenEnv needs both a new rollout abstraction and a more efficient attention implementation.

## Note

FlashAttention's cumulative sequence lengths, or varlen handling, are applied during both inference and training batch processing.